In [1]:
from Data_query.trino_config import *
import numpy as np
from visualisation import *
import pytz

In [6]:
stop_trino()

Trino service stopping triggered.


In [5]:
sleep(120)


In [2]:
big_workers = 1
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
# sleep(90)

Trino service is already running.


In [3]:
iceberg_exec("DROP TABLE IF EXISTS voltwatt_uncurtailedPV")
iceberg_exec("""CREATE TABLE voltwatt_uncurtailedPV (
                site_id BIGINT,
                t_stamp TIMESTAMP,
                uncurtailed_P DOUBLE,
                V DOUBLE
            )
    """)

Executed
Executed


In [5]:
sleep(30)

In [4]:
num_parts=2
time_bin_interval = '5'
model = "pv_ghi_norm_model"
acceptible_sites = ', '.join(map(str, pd.read_csv('mape<50_sites.csv')['site_id'].tolist()))
def run_func(args):
    year, month, part = args
    # time_filter = f"year = {year} and month = {month}"
    time_filter = f"year = {year}"
    part_filter = f"site_id % {num_parts} = {part}"
    df = iceberg_exec(f"""
                    insert into voltwatt_uncurtailedPV
                    with voltwatt_data AS (
                        SELECT
                            site_id,
                            actual_day,
                            t_stamp,
                            CAST(
                                date_trunc('minute', t_stamp + interval '10' hour)
                                - interval '1' minute * (minute(t_stamp + interval '10' hour) % {time_bin_interval})
                                AS TIME) AS tod_bin,
                            GHI/GHI_cs AS x,
                            GHI_cs,
                            P_kw_norm/ NULLIF(P_kw_norm_cs, 0.0) AS y,
                                P_kw_norm,
                                P_kw_norm_cs,
                                S_norm,
                                S_99,
                                V
                        FROM structured_data
                        WHERE P_kw_norm_cs > 0.2 AND GHI > 50 and P_kw_norm > 0.05 and P_kw_norm <= P_kw_norm_cs
                            and {time_filter} and {part_filter} and site_id in ({acceptible_sites}) and V > 253
                    ),
                    validation_on_voltwatt_data AS (
                        select 
                            t.site_id, 
                            t.t_stamp, 
                            x as GHI, 
                            P_kw_norm, 
                            case when P_kw_norm_cs * (a + b * x) >= P_kw_norm then P_kw_norm_cs * (a + b * x) else P_kw_norm end AS P_kw_norm_est,
                            V,
                            S_norm,
                            S_99
                        from voltwatt_data t 
                            join {model} m on t.site_id = m.site_id and t.tod_bin = m.tod_bin
                    )
                    SELECT site_id, t_stamp, P_kw_norm_est*S_99 as uncurtailed_P, V 
                    FROM validation_on_voltwatt_data
                        where P_kw_norm_est is not null
                        """)

                    # 

    sleep(10)
    print(f"Completed {time_filter}, part {part}")
    return df
tasks = [(year, month, part) for year in (2024, ) for month in range(1, 2) 
         for part in range(0, num_parts)]
            
try:         
    df = trino_parallel_batch(run_func, tasks, num_workers=num_workers, batch_size=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    # stop_trino()
    pass
# df['t_stamp'] = pd.to_datetime(df['t_stamp']).dt.tz_localize('utc').dt.tz_convert(pytz.FixedOffset(10*60))
df

Executed
Completed year = 2024, part 0
Executed
Completed year = 2024, part 1
